# PHASE 0 & 1: DATA LOADING & THE GREAT MERGER

In [10]:
import sys
import os
import pandas as pd

# Add the project root to the Python path to import 'src'
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import project configuration (Absolute paths)
from src import config

print("🧬 Integrating all processed datasets into one Master Intelligence Table...\n")

# 1. Load the processed datasets
df_mobility = pd.read_csv(os.path.join(config.PROCESSED_DATA_DIR, '01_mitma_demand_cleaned.csv'))
df_competition = pd.read_csv(os.path.join(config.PROCESSED_DATA_DIR, '02B_bcn_district_saturation.csv'))
df_income = pd.read_csv(os.path.join(config.PROCESSED_DATA_DIR, '03_bcn_income_by_district.csv'))

# 2. FIX MOBILITY KEYS: Rename and extract the exact district number
df_mobility = df_mobility.rename(columns={'district_id': 'Codi_Districte'})
# Convert "801901" into "1", "801910" into "10", etc.
df_mobility['Codi_Districte'] = df_mobility['Codi_Districte'].astype(str).str[-2:].astype(int)

# 3. Start the merge process using 'Codi_Districte' as the primary key
df_master = pd.merge(df_competition, df_income, on='Codi_Districte')

# 4. Clean up duplicate district name columns
if 'Nom_Districte_y' in df_master.columns:
    df_master = df_master.drop(columns=['Nom_Districte_y'])
if 'Nom_Districte_x' in df_master.columns:
    df_master = df_master.rename(columns={'Nom_Districte_x': 'Nom_Districte'})

# Final merge with Mobility data
df_master = pd.merge(df_master, df_mobility, on='Codi_Districte')

print(f"✅ Success! All datasets integrated. Master table has {len(df_master)} rows and {len(df_master.columns)} indicators.")

# 5. Display the Integrated Master Table
display(df_master)

🧬 Integrating all processed datasets into one Master Intelligence Table...

✅ Success! All datasets integrated. Master table has 10 rows and 6 indicators.


,Codi_Districte,Nom_Districte,total_competitors,saturation_pct,Renta_Media,daily_foot_traffic
0,2,Eixample,3061,27.584032,21976,1121181
1,1,Ciutat Vella,1588,14.310174,13990,374030
2,10,Sant Martí,1434,12.922411,17977,796929
3,3,Sants-Montjuïc,1064,9.588177,16911,706707
4,5,Sarrià-Sant Gervasi,849,7.650716,31710,603606
5,6,Gràcia,749,6.749572,21251,382325
6,9,Sant Andreu,665,5.992611,16979,531179
7,8,Nou Barris,647,5.830405,13964,460691
8,4,Les Corts,524,4.721997,26069,400509
9,7,Horta-Guinardó,516,4.649905,17234,513710


# PHASE 2: FEATURE ENGINEERING - THE OPPORTUNITY SCORE

In [13]:
print("🧠 Calculating the final Opportunity Score for each district...\n")

# 1. Normalize the variables on a scale from 0 to 1 (Min-Max Scaling)
# Traffic (More is better)
min_traffic = df_master['daily_foot_traffic'].min()
max_traffic = df_master['daily_foot_traffic'].max()
df_master['Traffic_Norm'] = (df_master['daily_foot_traffic'] - min_traffic) / (max_traffic - min_traffic)

# Income (More is better)
min_income = df_master['Renta_Media'].min()
max_income = df_master['Renta_Media'].max()
df_master['Income_Norm'] = (df_master['Renta_Media'] - min_income) / (max_income - min_income)

# Competition (Less is better, so we will invert it later in the formula)
# FIX: Using 'total_competitors' instead of 'Total_Restaurants'
min_comp = df_master['total_competitors'].min()
max_comp = df_master['total_competitors'].max()
df_master['Competition_Norm'] = (df_master['total_competitors'] - min_comp) / (max_comp - min_comp)

# 2. The Business Algorithm: Calculate the Opportunity Score (0 to 100)
# Weights: 40% Traffic, 40% Income, 20% Lack of Competition (1 - Competition_Norm)
df_master['Opportunity_Score'] = (
    (df_master['Traffic_Norm'] * 0.40) + 
    (df_master['Income_Norm'] * 0.40) + 
    ((1 - df_master['Competition_Norm']) * 0.20)
) * 100

# Round the score to 2 decimal places for cleaner visualization
df_master['Opportunity_Score'] = df_master['Opportunity_Score'].round(2)

# 3. Sort the dataframe to reveal the "Gold Mine" at the top
df_master = df_master.sort_values(by='Opportunity_Score', ascending=False).reset_index(drop=True)

# 4. Display the Final Ranking
columns_to_show = ['Nom_Districte', 'daily_foot_traffic', 'Renta_Media', 'total_competitors', 'Opportunity_Score']
display(df_master[columns_to_show])

🧠 Calculating the final Opportunity Score for each district...



,Nom_Districte,daily_foot_traffic,Renta_Media,total_competitors,Opportunity_Score
0,Sarrià-Sant Gervasi,603606,31710,849,69.67
1,Eixample,1121181,21976,3061,58.06
2,Les Corts,400509,26069,524,48.64
3,Sant Martí,796929,17977,1434,44.47
4,Sants-Montjuïc,706707,16911,1064,40.15
5,Gràcia,382325,21251,749,35.04
6,Horta-Guinardó,513710,17234,516,34.85
7,Sant Andreu,531179,16979,665,34.04
8,Nou Barris,460691,13964,647,23.61
9,Ciutat Vella,374030,13990,1588,11.63
